# 06a · LSTM 경보에 대한 JEV confirmation gate

Fault 3의 **저장된 test prediction**만 사용하는 24행 사후 파일럿이다. JEV는 LSTM이 낸 5,000초 이내 경보의 유지 근거를 3단계로 평가한다. JEV가 새 TTF를 예측하지 않는다. 임계값과 샘플 위치는 결과 확인 전에 5,000초, native score 1.0, sequence별 10/50/90%로 고정한다.

In [ ]:
from pathlib import Path
import hashlib, json, math, os, time, urllib.error, urllib.request
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path(r"C:\projects\FabJudge-PHM2018-starter\FabJudge-PHM2018")
PREDICTIONS = ROOT / "artifacts/huang2018/fault3_test_predictions.parquet"
SPLIT = ROOT / "artifacts/huang2018/split_metadata.json"
OUT = ROOT / "artifacts/06a_lstm_jev"
CACHE = OUT / "cache"
THRESHOLD_SECONDS = 5000
MODEL = "typesafe/jev-1.13"
URL = "https://openrouter.ai/api/alpha/decisions"
TIMEOUT_SECONDS = 60
MAX_NEW_CALLS = 7

predictions = pd.read_parquet(PREDICTIONS)
metadata = json.loads(SPLIT.read_text(encoding="utf-8"))
required = {"sequence_id", "raw_row_index", "lstm_pred_seconds", "rf_probability",
            "pipeline_rul_seconds", "wall_ttf_seconds"}
missing_columns = required - set(predictions.columns)
if missing_columns:
    raise ValueError(f"Prediction columns missing: {sorted(missing_columns)}")
metadata_sequences = set(metadata["splits"]["fault3"]["test_original_sequences"])
prediction_sequences = set(predictions["sequence_id"].unique())
metadata_only = sorted(metadata_sequences - prediction_sequences)
prediction_only = sorted(prediction_sequences - metadata_sequences)
print({"split_exact_match": not (metadata_only or prediction_only),
       "metadata_only": metadata_only, "prediction_only": prediction_only})
if metadata_only or prediction_only:
    raise ValueError("Fault 3 test sequence mismatch; JEV calls stopped")
if predictions.duplicated(["sequence_id", "raw_row_index"]).any():
    raise ValueError("Duplicate sequence_id/raw_row_index pairs")

## 위치만으로 샘플 선택

각 sequence의 `raw_row_index` 오름차순에서 `floor(p × (n−1))`를 0부터 시작하는 위치로 사용한다 (`p=0.1, 0.5, 0.9`). 이 단계는 sequence ID를 **그룹 키**로만 사용하고, ID 문자열의 고장 시각을 해석하지 않는다. 정답 열과 prediction 값은 위치 결정에 사용하지 않는다.

In [ ]:
positions = [("10%", 0.1), ("50%", 0.5), ("90%", 0.9)]
ordered_keys = predictions[["sequence_id", "raw_row_index"]].sort_values(
    ["sequence_id", "raw_row_index"], kind="stable")
chosen = []
for _, group in ordered_keys.groupby("sequence_id", sort=True):
    indices = [math.floor(p * (len(group) - 1)) for _, p in positions]
    part = group.iloc[indices].copy()
    part["sample_position"] = [label for label, _ in positions]
    chosen.append(part)
sampled_keys = pd.concat(chosen, ignore_index=True)
if len(metadata_sequences) != 8 or len(sampled_keys) != 24 or sampled_keys.duplicated(
        ["sequence_id", "raw_row_index"]).any():
    raise ValueError(f"Expected 8 sequences and 24 distinct rows; got {len(metadata_sequences)} and {len(sampled_keys)}")
print({"pilot_rows": len(sampled_keys), "sequences": sampled_keys.sequence_id.nunique(),
       "position_counts": sampled_keys.sample_position.value_counts().to_dict()})

# Ground truth and both fixed baseline alarms are created only after selection is complete.
pilot = sampled_keys.merge(predictions, on=["sequence_id", "raw_row_index"],
                           how="left", validate="one_to_one", sort=False)
pilot["actual_fail_5000"] = pilot["wall_ttf_seconds"] <= THRESHOLD_SECONDS
pilot["lstm_alarm"] = pilot["lstm_pred_seconds"] <= THRESHOLD_SECONDS
pilot["pipeline_alarm"] = pilot["pipeline_rul_seconds"] <= THRESHOLD_SECONDS
pilot["jev_called"] = False
pilot["jev_status"] = np.where(pilot.lstm_alarm, "not_attempted", "not_called")
pilot["jev_score"] = np.nan
pilot["jev_keep"] = pd.Series(pd.NA, index=pilot.index, dtype="boolean")
pilot["hybrid_alarm"] = pd.Series(pd.NA, index=pilot.index, dtype="boolean")
pilot.loc[~pilot.lstm_alarm, "hybrid_alarm"] = False
pilot["jev_error"] = None
for col in ("recent_lstm_delta", "recent_lstm_std", "history_count"):
    pilot[col] = np.nan
alarm_count = int(pilot.lstm_alarm.sum())
print({"lstm_alarm_rows": alarm_count, "jev_target_rows": alarm_count,
       "expected_target_rows": 7, "matches_expectation": alarm_count == 7})
if alarm_count > MAX_NEW_CALLS:
    raise ValueError(f"{alarm_count} targets exceed the fixed {MAX_NEW_CALLS}-call limit")

## JEV 입력 상태

경보 행의 같은 sequence에서 현재 행까지의 **최근 최대 5개** LSTM prediction만 사용한다. `recent_lstm_delta = 현재 prediction − 사용된 history의 가장 오래된 prediction`이며, 표준편차는 이 history의 모집단 표준편차 (`ddof=0`)다. sequence ID와 평가용 정답, pipeline 예측은 JEV 요청에 넣지 않는다.

In [ ]:
history_source = predictions[["sequence_id", "raw_row_index", "lstm_pred_seconds"]].sort_values(
    ["sequence_id", "raw_row_index"], kind="stable")
histories = {sid: group.reset_index(drop=True) for sid, group in
             history_source.groupby("sequence_id", sort=False)}
for row_index, row in pilot.loc[pilot.lstm_alarm].iterrows():
    group = histories[row.sequence_id]
    sorted_indices = group.raw_row_index.to_numpy()
    current_position = int(np.searchsorted(sorted_indices, row.raw_row_index))
    if current_position >= len(group) or sorted_indices[current_position] != row.raw_row_index:
        raise ValueError("Alarm row missing from its sequence history")
    values = group.lstm_pred_seconds.iloc[max(0, current_position - 4):current_position + 1].to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError("Non-finite LSTM history")
    pilot.at[row_index, "recent_lstm_delta"] = float(values[-1] - values[0])
    pilot.at[row_index, "recent_lstm_std"] = float(np.std(values, ddof=0))
    pilot.at[row_index, "history_count"] = len(values)

QUESTION = {
    "type": "score",
    "instructions": "Using only state.features, how strong is the evidence to keep the current LSTM alarm for failure within 5000 seconds? Assess the alarm evidence; do not estimate a new time to failure.",
    "criteria": [
        "Weak: the available LSTM trend and RF signal provide weak support for keeping the alarm",
        "Uncertain: the available signals give mixed or insufficient support for keeping the alarm",
        "Strong: the available LSTM trend and RF signal provide strong support for keeping the alarm",
    ],
}

def compact_state(row):
    state = {"lstm_pred_seconds": float(row.lstm_pred_seconds),
             "rf_probability": float(row.rf_probability),
             "recent_lstm_delta": float(row.recent_lstm_delta),
             "recent_lstm_std": float(row.recent_lstm_std),
             "history_count": int(row.history_count)}
    if any(not math.isfinite(value) for value in state.values()):
        raise ValueError("Non-finite JEV state")
    return state

def request_spec(row):
    return {"endpoint": URL, "method": "POST", "timeout_seconds": TIMEOUT_SECONDS,
            "body": {"model": MODEL, "state": {"features": compact_state(row)},
                     "questions": {"alarm_evidence": QUESTION}}}

for _, row in pilot.loc[pilot.lstm_alarm].iterrows():
    spec = request_spec(row)
    assert set(spec["body"]["state"]["features"]) == {
        "lstm_pred_seconds", "rf_probability", "recent_lstm_delta", "recent_lstm_std", "history_count"}

## OpenRouter Decisions API

`05a`와 같은 Decisions endpoint, 환경 변수/`.env` credential 방식, score 응답 검증, 60초 timeout을 사용한다. 첫 대상의 응답을 검증한 직후 모델, provider, score 형식을 확인한다. 첫 응답이 실패하면 나머지는 호출하지 않는다. 성공 score 1.0 이상만 LSTM 경보를 유지한다.

In [ ]:
def load_api_key():
    values = {}
    env_file = ROOT / ".env"
    if env_file.is_file():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            s = line.strip()
            if s and not s.startswith("#") and "=" in s:
                name, value = s.split("=", 1)
                name, value = name.strip(), value.strip().strip('"').strip("'")
                if name in {"JEV_API_KEY", "OPENROUTER_API_KEY"} and value:
                    values[name] = value
    return (os.environ.get("JEV_API_KEY") or values.get("JEV_API_KEY")
            or os.environ.get("OPENROUTER_API_KEY") or values.get("OPENROUTER_API_KEY"))

API_KEY = load_api_key()  # Never display or serialize this value.
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      separators=(",", ":"), allow_nan=False)

def validate_response(raw):
    if not isinstance(raw, dict):
        raise ValueError("Response is not an object")
    model, provider = raw.get("model"), raw.get("provider")
    if not isinstance(model, str) or not model.startswith(MODEL):
        raise ValueError("Unexpected response model")
    if provider != "TypeSafe":
        raise ValueError("Unexpected response provider")
    answers = raw.get("answers")
    if not isinstance(answers, dict) or set(answers) != {"alarm_evidence"}:
        raise ValueError("Unexpected answers schema")
    answer = answers["alarm_evidence"]
    if not isinstance(answer, dict) or answer.get("type") != "score":
        raise ValueError("Unexpected response format")
    score = answer.get("score")
    if isinstance(score, bool) or not isinstance(score, (int, float)) or not math.isfinite(score) or not 0 <= score <= 2:
        raise ValueError("Missing, non-numeric, or out-of-range native score")
    confidence = answer.get("confidence")
    if isinstance(confidence, bool) or not isinstance(confidence, (int, float)) or not math.isfinite(confidence) or not 0 <= confidence <= 1:
        raise ValueError("Invalid confidence")
    probabilities = answer.get("probabilities")
    if not isinstance(probabilities, dict) or set(probabilities) != {"0", "1", "2"}:
        raise ValueError("Unexpected probabilities schema")
    if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) or not 0 <= v <= 1
           for v in probabilities.values()) or not .95 <= sum(probabilities.values()) <= 1.05:
        raise ValueError("Invalid probabilities")
    legend = answer.get("legend")
    if not isinstance(legend, dict) or legend != {str(i): text for i, text in enumerate(QUESTION["criteria"])}:
        raise ValueError("Unexpected 3-level criterion legend")
    return float(score)

new_calls = 0
cache_hits = 0

def call_jev(row):
    global new_calls, cache_hits
    spec = request_spec(row)
    cache_path = CACHE / (hashlib.sha256(canonical(spec).encode("utf-8")).hexdigest() + ".json")
    if cache_path.is_file():
        cache_hits += 1
        try:
            record = json.loads(cache_path.read_text(encoding="utf-8"))
            if record.get("request_sha256") != cache_path.stem:
                raise ValueError("Cache request hash mismatch")
            raw = record.get("response")
            score = validate_response(raw) if raw is not None else None
            return {"status": "cached" if score is not None else "failed", "score": score,
                    "error": record.get("error"), "response": raw, "cache_hit": True}
        except (ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
            return {"status": "failed", "score": None, "error": f"Invalid cache: {type(exc).__name__}: {exc}",
                    "response": None, "cache_hit": True}
    if not API_KEY:
        return {"status": "failed", "score": None, "error": "OpenRouter credential unavailable",
                "response": None, "cache_hit": False}
    if new_calls >= MAX_NEW_CALLS:
        return {"status": "failed", "score": None, "error": "New API call cap reached",
                "response": None, "cache_hit": False}
    request = urllib.request.Request(URL,
        data=canonical(spec["body"]).encode("utf-8"),
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}, method="POST")
    new_calls += 1
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=TIMEOUT_SECONDS) as response:
            api_raw = json.loads(response.read().decode("utf-8"))
        # Retain only the fields needed to validate and account for this decision.
        raw = {name: api_raw.get(name) for name in ("model", "provider", "answers", "usage")}
        if API_KEY in canonical(raw):
            raise ValueError("Credential appeared in response; refusing to cache")
        score = validate_response(raw)
        status, error = "success", None
    except urllib.error.HTTPError as exc:
        raw, score, status, error = None, None, "failed", f"HTTP {exc.code}"
    except Exception as exc:
        raw, score, status = None, None, "failed"
        error = f"{type(exc).__name__}: {str(exc).replace(API_KEY, '[REDACTED]')}"
    record = {"request_sha256": cache_path.stem, "response": raw, "error": error,
              "latency_seconds": time.perf_counter() - started}
    if status == "success":
        cache_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status": status, "score": score, "error": error,
            "response": raw, "cache_hit": False}

def set_result(index, result):
    pilot.at[index, "jev_called"] = (not result["cache_hit"] and result["error"] not in
        {"OpenRouter credential unavailable", "New API call cap reached"})
    pilot.at[index, "jev_status"] = result["status"]
    pilot.at[index, "jev_error"] = result["error"]
    if result["score"] is not None:
        pilot.at[index, "jev_score"] = result["score"]
        keep = result["score"] >= 1.0
        pilot.at[index, "jev_keep"] = keep
        pilot.at[index, "hybrid_alarm"] = bool(pilot.at[index, "lstm_alarm"] and keep)

alarm_indices = pilot.index[pilot.lstm_alarm].tolist()
responses = {}
if alarm_indices:
    first_index = alarm_indices[0]
    first_result = call_jev(pilot.loc[first_index])
    responses[first_index] = first_result
    set_result(first_index, first_result)
    first_raw = first_result["response"] or {}
    first_answer = (first_raw.get("answers") or {}).get("alarm_evidence") or {}
    sanity = {"model": first_raw.get("model"), "provider": first_raw.get("provider"),
              "response_format": first_answer.get("type"), "native_score_present": "score" in first_answer,
              "score_numeric": isinstance(first_answer.get("score"), (int, float)) and not isinstance(first_answer.get("score"), bool),
              "schema_valid": first_result["score"] is not None, "error": first_result["error"]}
    print("First response sanity check:", sanity)
    if first_result["score"] is not None:
        for index in alarm_indices[1:]:
            result = call_jev(pilot.loc[index])
            responses[index] = result
            set_result(index, result)
    else:
        print("First response failed validation; remaining JEV calls stopped.")
assert new_calls <= MAX_NEW_CALLS

## 동일 24행 비교와 저장

JEV 실패 또는 미시도 경보의 hybrid 값은 missing으로 둔다. 이 경우 24행 전체의 hybrid confusion matrix는 계산하지 않으며 실패와 미시도 건수를 별도로 기록한다. Precision의 분모가 0이면 `null`로 기록한다. Pipeline 예측의 missing 값은 고정된 `<= 5000` 식에 따라 경보 없음으로 처리된다.

In [ ]:
def metrics(name, predicted):
    missing = int(pd.isna(predicted).sum())
    result = {"system": name, "evaluation_rows": len(pilot), "unresolved_rows": missing}
    if missing:
        return {**result, "TP": None, "FP": None, "FN": None, "TN": None,
                "precision": None, "recall": None}
    p = pd.Series(predicted, index=pilot.index).astype(bool)
    y = pilot.actual_fail_5000.astype(bool)
    tp, fp = int((p & y).sum()), int((p & ~y).sum())
    fn, tn = int((~p & y).sum()), int((~p & ~y).sum())
    return {**result, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": tp / (tp + fp) if tp + fp else None,
            "recall": tp / (tp + fn) if tp + fn else None}

metric_records = [
    metrics("LSTM only", pilot.lstm_alarm),
    metrics("RF to LSTM pipeline", pilot.pipeline_alarm),
    metrics("LSTM + JEV confirmation gate", pilot.hybrid_alarm),
]
comparison = pd.DataFrame(metric_records)
rejected = pilot.loc[pilot.lstm_alarm & pilot.jev_keep.fillna(False).eq(False) & pilot.jev_score.notna()]
removed_false_positives = int((~rejected.actual_fail_5000).sum())
removed_true_positives = int(rejected.actual_fail_5000.sum())

def reported_sum(items, field):
    if not items:
        return "unavailable"
    values = []
    for item in items:
        usage = (item.get("response") or {}).get("usage") or {}
        value = usage.get(field)
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            return "unavailable"
        values.append(float(value))
    return sum(values)

responded = [r for r in responses.values() if r["response"] is not None]
new_responded = [r for r in responded if not r["cache_hit"]]
summary = {
    "pilot_rows": len(pilot), "test_sequences": len(metadata_sequences),
    "lstm_alarm_rows": alarm_count, "jev_target_rows": alarm_count,
    "new_api_calls": new_calls, "cache_hits": cache_hits,
    "jev_success_rows": int(pilot.jev_status.isin(["success", "cached"]).sum()),
    "jev_failed_rows": int(pilot.jev_status.eq("failed").sum()),
    "jev_not_attempted_rows": int(pilot.jev_status.eq("not_attempted").sum()),
    "api_usage": {"pilot_input_tokens": reported_sum(responded, "input_tokens"),
                  "pilot_output_tokens": reported_sum(responded, "output_tokens"),
                  "pilot_reported_cost_usd": reported_sum(responded, "cost"),
                  "new_calls_reported_cost_usd": 0.0 if new_calls == 0 else reported_sum(new_responded, "cost")},
    "metrics": metric_records,
    "removed_false_positives": removed_false_positives,
    "removed_true_positives": removed_true_positives,
    "hybrid_comparison_complete": bool(pilot.hybrid_alarm.notna().all()),
}
row_columns = ["sequence_id", "raw_row_index", "sample_position", "lstm_pred_seconds",
               "rf_probability", "recent_lstm_delta", "recent_lstm_std", "history_count",
               "lstm_alarm", "pipeline_alarm", "jev_called", "jev_score", "jev_status",
               "jev_keep", "hybrid_alarm", "actual_fail_5000", "jev_error"]
pilot[row_columns].to_csv(OUT / "pilot_rows.csv", index=False)
comparison.to_csv(OUT / "metric_comparison.csv", index=False)
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
display(comparison)
print({key: summary[key] for key in ("new_api_calls", "cache_hits", "jev_success_rows",
      "jev_failed_rows", "jev_not_attempted_rows", "api_usage")})
if summary["hybrid_comparison_complete"]:
    print(f"Pilot conclusion: JEV removed {removed_false_positives} false alarms and {removed_true_positives} true alarms from the fixed LSTM alarms.")
else:
    print("Pilot conclusion: Hybrid comparison on all 24 rows is unavailable because JEV decisions are unresolved.")
    print(f"Resolved rejections so far: {removed_false_positives} false alarms, {removed_true_positives} true alarms.")

## 후속 실험 · 보수적 confirmation prompt 1개

기존 24행 결과를 본 **이후** 작성한 탐색적 prompt다. 같은 7개 LSTM 경보에만 JEV를 다시 호출한다. 모델, 상태 숫자 5개, 3단계 criteria, 5,000초와 native score 1.0 임계값은 그대로 둔다. 새 instructions는 낮은 RF 값 또는 짧은 LSTM 추세 하나만으로 경보를 거절하지 않고, 증거가 혼재하거나 부족하면 `Uncertain`을 택하도록 유도한다. 정답은 요청에 넣지 않는다. 이 비교는 독립 검증 또는 일반화 성능의 증거가 아니다.

In [ ]:
# Read the original 06a results without changing its artifacts.
BASELINE_ROWS = pd.read_csv(OUT / "pilot_rows.csv")
BASELINE_SUMMARY = json.loads((OUT / "summary.json").read_text(encoding="utf-8"))
if len(BASELINE_ROWS) != 24 or int(BASELINE_ROWS.lstm_alarm.sum()) != 7:
    raise ValueError("Expected the original fixed 24-row, 7-alarm pilot")
TUNE_OUT = OUT / "prompt_tune_01"
TUNE_CACHE = TUNE_OUT / "cache"
TUNE_OUT.mkdir(parents=True, exist_ok=True)
TUNE_CACHE.mkdir(parents=True, exist_ok=True)
TUNED_QUESTION = {
    "type": QUESTION["type"],
    "instructions": (QUESTION["instructions"] + " This is a conservative confirmation gate. "
        "Choose Weak only when the available signals clearly and jointly contradict keeping the alarm. "
        "A low RF signal or a flat or rising short LSTM trend alone is not enough to reject it. "
        "When evidence is mixed or limited, choose Uncertain."),
    "criteria": list(QUESTION["criteria"]),
}
assert TUNED_QUESTION["criteria"] == QUESTION["criteria"] and TUNED_QUESTION["type"] == "score"
(TUNE_OUT / "decision_question.json").write_text(
    json.dumps(TUNED_QUESTION, ensure_ascii=False, indent=2), encoding="utf-8")

tuned = BASELINE_ROWS.copy()
tuned["tuned_jev_score"] = np.nan
tuned["tuned_jev_status"] = np.where(tuned.lstm_alarm, "not_attempted", "not_called")
tuned["tuned_jev_called"] = False
tuned["tuned_cache_hit"] = False
tuned["tuned_jev_error"] = None
tuned["tuned_jev_keep"] = pd.Series(pd.NA, index=tuned.index, dtype="boolean")
tuned["tuned_hybrid_alarm"] = pd.Series(pd.NA, index=tuned.index, dtype="boolean")
tuned.loc[~tuned.lstm_alarm, "tuned_hybrid_alarm"] = False

# Existing 05a-style call/validator functions use these two globals. Restore them after the trial.
ORIGINAL_QUESTION, ORIGINAL_CACHE = QUESTION, CACHE
QUESTION, CACHE = TUNED_QUESTION, TUNE_CACHE
new_calls, cache_hits = 0, 0
tune_responses = {}
try:
    for trial_number, index in enumerate(tuned.index[tuned.lstm_alarm]):
        before = new_calls
        result = call_jev(tuned.loc[index])
        tune_responses[index] = result
        tuned.at[index, "tuned_jev_called"] = new_calls > before
        tuned.at[index, "tuned_cache_hit"] = result["cache_hit"]
        tuned.at[index, "tuned_jev_status"] = result["status"]
        tuned.at[index, "tuned_jev_error"] = result["error"]
        if result["score"] is not None:
            tuned.at[index, "tuned_jev_score"] = result["score"]
            keep = result["score"] >= 1.0
            tuned.at[index, "tuned_jev_keep"] = keep
            tuned.at[index, "tuned_hybrid_alarm"] = keep
        if trial_number == 0:
            first_raw = result["response"] or {}
            first_answer = (first_raw.get("answers") or {}).get("alarm_evidence") or {}
            print("Tuned first response:", {"model": first_raw.get("model"),
                "provider": first_raw.get("provider"), "response_format": first_answer.get("type"),
                "native_score_present": "score" in first_answer,
                "schema_valid": result["score"] is not None, "error": result["error"]})
            if result["score"] is None:
                print("First tuned response failed; remaining calls stopped.")
                break
finally:
    QUESTION, CACHE = ORIGINAL_QUESTION, ORIGINAL_CACHE
assert new_calls <= 7
print({"tuned_new_api_calls": new_calls, "tuned_cache_hits": cache_hits,
       "tuned_success_rows": int(tuned.tuned_jev_status.isin(["success", "cached"]).sum()),
       "tuned_failed_rows": int(tuned.tuned_jev_status.eq("failed").sum())})

In [ ]:
def tuned_metrics(predicted):
    unresolved = int(pd.isna(predicted).sum())
    result = {"system": "LSTM + JEV conservative prompt", "evaluation_rows": 24,
              "unresolved_rows": unresolved}
    if unresolved:
        return {**result, "TP": None, "FP": None, "FN": None, "TN": None,
                "precision": None, "recall": None}
    p = pd.Series(predicted, index=tuned.index).astype(bool)
    y = tuned.actual_fail_5000.astype(bool)
    tp, fp = int((p & y).sum()), int((p & ~y).sum())
    fn, tn = int((~p & y).sum()), int((~p & ~y).sum())
    return {**result, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": tp / (tp + fp) if tp + fp else None,
            "recall": tp / (tp + fn) if tp + fn else None}

def usage_sum(response_records, field):
    if not response_records:
        return "unavailable"
    values = []
    for record in response_records:
        value = ((record.get("response") or {}).get("usage") or {}).get(field)
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            return "unavailable"
        values.append(float(value))
    return sum(values)

tuned_metric = tuned_metrics(tuned.tuned_hybrid_alarm)
metric_records_tuned = BASELINE_SUMMARY["metrics"] + [tuned_metric]
comparison_tuned = pd.DataFrame(metric_records_tuned)
rejected_tuned = tuned.loc[tuned.lstm_alarm & tuned.tuned_jev_score.notna() &
                           tuned.tuned_jev_keep.fillna(False).eq(False)]
removed_fp = int((~rejected_tuned.actual_fail_5000).sum())
removed_tp = int(rejected_tuned.actual_fail_5000.sum())
responded = [record for record in tune_responses.values() if record["response"] is not None]
new_responded = [record for record in responded if not record["cache_hit"]]
tune_summary = {
    "design": "single post-hoc conservative prompt on the original fixed 24 rows",
    "question": TUNED_QUESTION,
    "pilot_rows": 24, "lstm_alarm_rows": 7,
    "new_api_calls": new_calls, "cache_hits": cache_hits,
    "jev_success_rows": int(tuned.tuned_jev_status.isin(["success", "cached"]).sum()),
    "jev_failed_rows": int(tuned.tuned_jev_status.eq("failed").sum()),
    "jev_not_attempted_rows": int(tuned.tuned_jev_status.eq("not_attempted").sum()),
    "api_usage": {"input_tokens": usage_sum(responded, "input_tokens"),
                  "output_tokens": usage_sum(responded, "output_tokens"),
                  "pilot_reported_cost_usd": usage_sum(responded, "cost"),
                  "new_calls_reported_cost_usd": 0.0 if new_calls == 0 else usage_sum(new_responded, "cost")},
    "metrics": metric_records_tuned,
    "removed_false_positives": removed_fp, "removed_true_positives": removed_tp,
    "hybrid_comparison_complete": bool(tuned.tuned_hybrid_alarm.notna().all()),
}
row_columns = ["sequence_id", "raw_row_index", "sample_position", "lstm_pred_seconds",
               "rf_probability", "recent_lstm_delta", "recent_lstm_std", "history_count",
               "lstm_alarm", "jev_score", "hybrid_alarm", "tuned_jev_score", "tuned_jev_status",
               "tuned_jev_called", "tuned_cache_hit", "tuned_jev_keep", "tuned_hybrid_alarm",
               "actual_fail_5000", "tuned_jev_error"]
tuned[row_columns].to_csv(TUNE_OUT / "pilot_rows.csv", index=False)
comparison_tuned.to_csv(TUNE_OUT / "metric_comparison.csv", index=False)
(TUNE_OUT / "summary.json").write_text(
    json.dumps(tune_summary, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
display(comparison_tuned)
print({"removed_false_positives": removed_fp, "removed_true_positives": removed_tp,
       "new_api_calls": new_calls, "cache_hits": cache_hits, "api_usage": tune_summary["api_usage"]})
if tune_summary["hybrid_comparison_complete"]:
    print("Exploratory post-hoc comparison complete; these same 24 rows cannot validate prompt generalization.")
else:
    print("Tuned comparison remains incomplete because one or more JEV decisions are unresolved.")

## 후속 실험 · 더 약한 prompt 변형

첫 보수적 prompt는 true alarm을 모두 유지했지만 false alarm도 유지했다. 아래 변형은 instructions에 **RF 신호와 LSTM 경보의 불일치만으로 거절하지 말고 현재 LSTM 예측과 최근 이력도 함께 고려하라**는 한 문장만 추가한다. criteria, 상태 숫자, 임계값은 다시 고정한다. 같은 test 행을 반복 사용한 사후 탐색이며, 두 후보 중 더 좋아 보이는 결과를 독립 검증 성능으로 해석하지 않는다.

In [ ]:
TUNED2_QUESTION = {
    "type": ORIGINAL_QUESTION["type"],
    "instructions": (ORIGINAL_QUESTION["instructions"] +
        " Because this is a confirmation gate, do not treat RF disagreement alone as sufficient to reject an LSTM alarm; weigh the current LSTM prediction and its recent history before rating support as Weak."),
    "criteria": list(ORIGINAL_QUESTION["criteria"]),
}
assert TUNED2_QUESTION["criteria"] == ORIGINAL_QUESTION["criteria"]
TUNE2_OUT = OUT / "prompt_tune_02"
TUNE2_CACHE = TUNE2_OUT / "cache"
TUNE2_OUT.mkdir(parents=True, exist_ok=True)
TUNE2_CACHE.mkdir(parents=True, exist_ok=True)
(TUNE2_OUT / "decision_question.json").write_text(
    json.dumps(TUNED2_QUESTION, ensure_ascii=False, indent=2), encoding="utf-8")

tuned2 = BASELINE_ROWS.copy()
tuned2["tuned2_jev_score"] = np.nan
tuned2["tuned2_jev_status"] = np.where(tuned2.lstm_alarm, "not_attempted", "not_called")
tuned2["tuned2_jev_called"] = False
tuned2["tuned2_cache_hit"] = False
tuned2["tuned2_jev_error"] = None
tuned2["tuned2_jev_keep"] = pd.Series(pd.NA, index=tuned2.index, dtype="boolean")
tuned2["tuned2_hybrid_alarm"] = pd.Series(pd.NA, index=tuned2.index, dtype="boolean")
tuned2.loc[~tuned2.lstm_alarm, "tuned2_hybrid_alarm"] = False
QUESTION, CACHE = TUNED2_QUESTION, TUNE2_CACHE
new_calls, cache_hits = 0, 0
tune2_responses = {}
try:
    for trial_number, index in enumerate(tuned2.index[tuned2.lstm_alarm]):
        before = new_calls
        result = call_jev(tuned2.loc[index])
        tune2_responses[index] = result
        tuned2.at[index, "tuned2_jev_called"] = new_calls > before
        tuned2.at[index, "tuned2_cache_hit"] = result["cache_hit"]
        tuned2.at[index, "tuned2_jev_status"] = result["status"]
        tuned2.at[index, "tuned2_jev_error"] = result["error"]
        if result["score"] is not None:
            tuned2.at[index, "tuned2_jev_score"] = result["score"]
            keep = result["score"] >= 1.0
            tuned2.at[index, "tuned2_jev_keep"] = keep
            tuned2.at[index, "tuned2_hybrid_alarm"] = keep
        if trial_number == 0:
            first_raw = result["response"] or {}
            first_answer = (first_raw.get("answers") or {}).get("alarm_evidence") or {}
            print("Tuned 2 first response:", {"model": first_raw.get("model"),
                "provider": first_raw.get("provider"), "response_format": first_answer.get("type"),
                "native_score_present": "score" in first_answer,
                "schema_valid": result["score"] is not None, "error": result["error"]})
            if result["score"] is None:
                print("First tuned 2 response failed; remaining calls stopped.")
                break
finally:
    QUESTION, CACHE = ORIGINAL_QUESTION, ORIGINAL_CACHE
assert new_calls <= 7

tuned2_metric = metrics("LSTM + JEV RF disagreement prompt", tuned2.tuned2_hybrid_alarm)
first_tune_summary = json.loads((TUNE_OUT / "summary.json").read_text(encoding="utf-8"))
metric_records_tuned2 = BASELINE_SUMMARY["metrics"] + [first_tune_summary["metrics"][-1], tuned2_metric]
comparison_tuned2 = pd.DataFrame(metric_records_tuned2)
rejected_tuned2 = tuned2.loc[tuned2.lstm_alarm & tuned2.tuned2_jev_score.notna() &
                             tuned2.tuned2_jev_keep.fillna(False).eq(False)]
removed_fp2 = int((~rejected_tuned2.actual_fail_5000).sum())
removed_tp2 = int(rejected_tuned2.actual_fail_5000.sum())
responded2 = [record for record in tune2_responses.values() if record["response"] is not None]
new_responded2 = [record for record in responded2 if not record["cache_hit"]]
tune2_summary = {
    "design": "second post-hoc prompt on the original fixed 24 rows",
    "question": TUNED2_QUESTION,
    "pilot_rows": 24, "lstm_alarm_rows": 7,
    "new_api_calls": new_calls, "cache_hits": cache_hits,
    "jev_success_rows": int(tuned2.tuned2_jev_status.isin(["success", "cached"]).sum()),
    "jev_failed_rows": int(tuned2.tuned2_jev_status.eq("failed").sum()),
    "jev_not_attempted_rows": int(tuned2.tuned2_jev_status.eq("not_attempted").sum()),
    "api_usage": {"input_tokens": usage_sum(responded2, "input_tokens"),
                  "output_tokens": usage_sum(responded2, "output_tokens"),
                  "pilot_reported_cost_usd": usage_sum(responded2, "cost"),
                  "new_calls_reported_cost_usd": 0.0 if new_calls == 0 else usage_sum(new_responded2, "cost")},
    "metrics": metric_records_tuned2,
    "removed_false_positives": removed_fp2, "removed_true_positives": removed_tp2,
    "hybrid_comparison_complete": bool(tuned2.tuned2_hybrid_alarm.notna().all()),
}
row_columns2 = ["sequence_id", "raw_row_index", "sample_position", "lstm_pred_seconds",
                "rf_probability", "recent_lstm_delta", "recent_lstm_std", "history_count",
                "lstm_alarm", "jev_score", "hybrid_alarm", "tuned2_jev_score", "tuned2_jev_status",
                "tuned2_jev_called", "tuned2_cache_hit", "tuned2_jev_keep", "tuned2_hybrid_alarm",
                "actual_fail_5000", "tuned2_jev_error"]
tuned2[row_columns2].to_csv(TUNE2_OUT / "pilot_rows.csv", index=False)
comparison_tuned2.to_csv(TUNE2_OUT / "metric_comparison.csv", index=False)
(TUNE2_OUT / "summary.json").write_text(
    json.dumps(tune2_summary, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
display(comparison_tuned2)
print({"removed_false_positives": removed_fp2, "removed_true_positives": removed_tp2,
       "new_api_calls": new_calls, "cache_hits": cache_hits, "api_usage": tune2_summary["api_usage"]})
if tune2_summary["hybrid_comparison_complete"]:
    print("Exploratory prompt comparison complete; independent validation is still required.")
else:
    print("Tuned 2 comparison remains incomplete because one or more JEV decisions are unresolved.")

## 더 큰 disjoint-row 후속 샘플

각 test sequence의 5, 15, ..., 95% 위치에서 10행씩 선택한다 (`floor(p × (n−1))`). 기존 10/50/90%의 24행과 겹치지 않는다. 선택과 overlap 검증에는 정답, prediction 값, sequence ID 내부의 고장 시각을 사용하지 않는다. 80행 선택 완료 후에만 정답을 만든다. 이 80행도 같은 8개 sequence에서 왔고 두 prompt는 기존 pilot 결과를 보고 만든 것이므로 독립적인 일반화 검증은 아니다.

In [ ]:
metadata_only = sorted(metadata_sequences - set(predictions.sequence_id.unique()))
prediction_only = sorted(set(predictions.sequence_id.unique()) - metadata_sequences)
print({"split_exact_match": not (metadata_only or prediction_only),
       "metadata_only": metadata_only, "prediction_only": prediction_only})
if metadata_only or prediction_only:
    raise ValueError("Test sequence mismatch; larger JEV test stopped")
large_positions = [(f"{percent}%", percent / 100) for percent in range(5, 100, 10)]
key_groups = []
for _, group in predictions[["sequence_id", "raw_row_index"]].sort_values(
        ["sequence_id", "raw_row_index"], kind="stable").groupby("sequence_id", sort=True):
    positions = [math.floor(fraction * (len(group) - 1)) for _, fraction in large_positions]
    selected_keys = group.iloc[positions].copy()
    selected_keys["sample_position"] = [label for label, _ in large_positions]
    key_groups.append(selected_keys)
large_keys = pd.concat(key_groups, ignore_index=True)
old_keys = pd.read_csv(OUT / "pilot_rows.csv", usecols=["sequence_id", "raw_row_index"])
overlap = len(large_keys.merge(old_keys, on=["sequence_id", "raw_row_index"]))
if len(metadata_sequences) != 8 or len(large_keys) != 80 or overlap or large_keys.duplicated(
        ["sequence_id", "raw_row_index"]).any():
    raise ValueError(f"Expected 80 distinct rows from 8 sequences and zero overlap; got {len(large_keys)}, {len(metadata_sequences)}, {overlap}")
print({"sample_rows": len(large_keys), "sequences": len(metadata_sequences),
       "position_counts": large_keys.sample_position.value_counts().sort_index().to_dict(),
       "overlap_with_24_row_pilot": overlap})

# Only now attach predictions and create the evaluation-only actual label.
large = large_keys.merge(predictions, on=["sequence_id", "raw_row_index"],
                         how="left", validate="one_to_one", sort=False)
large["actual_fail_5000"] = large.wall_ttf_seconds <= THRESHOLD_SECONDS
large["lstm_alarm"] = large.lstm_pred_seconds <= THRESHOLD_SECONDS
large["pipeline_alarm"] = large.pipeline_rul_seconds <= THRESHOLD_SECONDS
for column in ("recent_lstm_delta", "recent_lstm_std", "history_count"):
    large[column] = np.nan
for index, row in large.loc[large.lstm_alarm].iterrows():
    group = histories[row.sequence_id]
    indices = group.raw_row_index.to_numpy()
    position = int(np.searchsorted(indices, row.raw_row_index))
    if position >= len(group) or indices[position] != row.raw_row_index:
        raise ValueError("Alarm row missing from causal sequence history")
    values = group.lstm_pred_seconds.iloc[max(0, position - 4):position + 1].to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError("Non-finite LSTM history")
    large.at[index, "recent_lstm_delta"] = float(values[-1] - values[0])
    large.at[index, "recent_lstm_std"] = float(np.std(values, ddof=0))
    large.at[index, "history_count"] = len(values)
LARGE_ALARMS = int(large.lstm_alarm.sum())
MAX_LARGE_CALLS_PER_PROMPT = 23
print({"lstm_alarm_rows": LARGE_ALARMS, "candidate_count": 2,
       "maximum_new_api_calls": 2 * MAX_LARGE_CALLS_PER_PROMPT})
if LARGE_ALARMS != MAX_LARGE_CALLS_PER_PROMPT:
    raise ValueError(f"Expected 23 alarm rows before calls; found {LARGE_ALARMS}")

### 두 고정 prompt에만 JEV 호출

`prompt_tune_01`과 `prompt_tune_02`의 저장된 question을 그대로 읽는다. 각각 같은 23개 LSTM 경보에만 호출하고 전용 cache를 사용한다. 첫 응답 schema 검증에 실패하면 해당 후보의 후속 호출을 중단한다. API 요청에는 기존 compact state 숫자 5개만 들어간다.

In [ ]:
LARGE_OUT = OUT / "larger_test_01"
LARGE_OUT.mkdir(parents=True, exist_ok=True)
LARGE_QUESTIONS = {
    "conservative": json.loads((OUT / "prompt_tune_01/decision_question.json").read_text(encoding="utf-8")),
    "rf_disagreement": json.loads((OUT / "prompt_tune_02/decision_question.json").read_text(encoding="utf-8")),
}
for candidate, question in LARGE_QUESTIONS.items():
    if question["type"] != "score" or question["criteria"] != QUESTION["criteria"]:
        raise ValueError(f"Question specification changed for {candidate}")
(LARGE_OUT / "candidate_questions.json").write_text(
    json.dumps(LARGE_QUESTIONS, ensure_ascii=False, indent=2), encoding="utf-8")
large_keys.to_csv(LARGE_OUT / "sample_plan.csv", index=False)

def run_large_variant(candidate, enabled=True):
    global QUESTION, CACHE, MAX_NEW_CALLS, new_calls, cache_hits
    prefix = candidate
    large[f"{prefix}_jev_score"] = np.nan
    large[f"{prefix}_jev_status"] = np.where(large.lstm_alarm, "not_attempted", "not_called")
    large[f"{prefix}_jev_called"] = False
    large[f"{prefix}_cache_hit"] = False
    large[f"{prefix}_jev_error"] = None
    large[f"{prefix}_jev_keep"] = pd.Series(pd.NA, index=large.index, dtype="boolean")
    large[f"{prefix}_hybrid_alarm"] = pd.Series(pd.NA, index=large.index, dtype="boolean")
    large.loc[~large.lstm_alarm, f"{prefix}_hybrid_alarm"] = False
    result_set = {"new_api_calls": 0, "cache_hits": 0, "responses": {}, "first_response_valid": False}
    if not enabled:
        return result_set
    saved = (QUESTION, CACHE, MAX_NEW_CALLS, new_calls, cache_hits)
    QUESTION = LARGE_QUESTIONS[candidate]
    CACHE = LARGE_OUT / "cache" / candidate
    CACHE.mkdir(parents=True, exist_ok=True)
    MAX_NEW_CALLS, new_calls, cache_hits = MAX_LARGE_CALLS_PER_PROMPT, 0, 0
    try:
        for ordinal, index in enumerate(large.index[large.lstm_alarm]):
            before = new_calls
            response = call_jev(large.loc[index])
            result_set["responses"][index] = response
            large.at[index, f"{prefix}_jev_called"] = new_calls > before
            large.at[index, f"{prefix}_cache_hit"] = response["cache_hit"]
            large.at[index, f"{prefix}_jev_status"] = response["status"]
            large.at[index, f"{prefix}_jev_error"] = response["error"]
            if response["score"] is not None:
                large.at[index, f"{prefix}_jev_score"] = response["score"]
                keep = response["score"] >= 1.0
                large.at[index, f"{prefix}_jev_keep"] = keep
                large.at[index, f"{prefix}_hybrid_alarm"] = keep
            if ordinal == 0:
                raw = response["response"] or {}
                answer = (raw.get("answers") or {}).get("alarm_evidence") or {}
                result_set["first_response_valid"] = response["score"] is not None
                print(candidate, "first response:", {"model": raw.get("model"),
                    "provider": raw.get("provider"), "response_format": answer.get("type"),
                    "native_score_present": "score" in answer,
                    "schema_valid": result_set["first_response_valid"], "error": response["error"]})
                if not result_set["first_response_valid"]:
                    break
        result_set["new_api_calls"] = new_calls
        result_set["cache_hits"] = cache_hits
    finally:
        QUESTION, CACHE, MAX_NEW_CALLS, new_calls, cache_hits = saved
    return result_set

large_results = {"conservative": run_large_variant("conservative")}
large_results["rf_disagreement"] = run_large_variant(
    "rf_disagreement", enabled=large_results["conservative"]["first_response_valid"])
print({name: {"new_api_calls": result["new_api_calls"], "cache_hits": result["cache_hits"],
              "first_response_valid": result["first_response_valid"]}
       for name, result in large_results.items()})

### 80행 평가

LSTM 단독, 기존 RF→LSTM pipeline, 두 JEV confirmation gate를 동일 80행에서 비교한다. JEV 실패 또는 미시도 경보는 hybrid missing으로 남기고 전체 80행 혼동행렬을 계산하지 않는다. Precision 분모가 0이면 `null`이다. 결과는 두 prompt의 **같은 sequence에서의 추가 행**에 대한 관찰이다.

In [ ]:
def large_metrics(name, predicted):
    unresolved = int(pd.isna(predicted).sum())
    result = {"system": name, "evaluation_rows": len(large), "unresolved_rows": unresolved}
    if unresolved:
        return {**result, "TP": None, "FP": None, "FN": None, "TN": None,
                "precision": None, "recall": None}
    p = pd.Series(predicted, index=large.index).astype(bool)
    y = large.actual_fail_5000.astype(bool)
    tp, fp = int((p & y).sum()), int((p & ~y).sum())
    fn, tn = int((~p & y).sum()), int((~p & ~y).sum())
    return {**result, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": tp / (tp + fp) if tp + fp else None,
            "recall": tp / (tp + fn) if tp + fn else None}

large_metric_records = [large_metrics("LSTM only", large.lstm_alarm),
                        large_metrics("RF to LSTM pipeline", large.pipeline_alarm)]
large_candidate_summary = {}
for candidate, result in large_results.items():
    large_metric_records.append(large_metrics(f"LSTM + JEV {candidate}",
                                                 large[f"{candidate}_hybrid_alarm"]))
    rejected = large.loc[large.lstm_alarm & large[f"{candidate}_jev_score"].notna() &
                         large[f"{candidate}_jev_keep"].fillna(False).eq(False)]
    responded = [item for item in result["responses"].values() if item["response"] is not None]
    new_responded = [item for item in responded if not item["cache_hit"]]
    large_candidate_summary[candidate] = {
        "new_api_calls": result["new_api_calls"], "cache_hits": result["cache_hits"],
        "first_response_valid": result["first_response_valid"],
        "jev_success_rows": int(large[f"{candidate}_jev_status"].isin(["success", "cached"]).sum()),
        "jev_failed_rows": int(large[f"{candidate}_jev_status"].eq("failed").sum()),
        "jev_not_attempted_rows": int(large[f"{candidate}_jev_status"].eq("not_attempted").sum()),
        "removed_false_positives": int((~rejected.actual_fail_5000).sum()),
        "removed_true_positives": int(rejected.actual_fail_5000.sum()),
        "api_usage": {"input_tokens": usage_sum(responded, "input_tokens"),
                      "output_tokens": usage_sum(responded, "output_tokens"),
                      "pilot_reported_cost_usd": usage_sum(responded, "cost"),
                      "new_calls_reported_cost_usd": 0.0 if result["new_api_calls"] == 0 else usage_sum(new_responded, "cost")},
    }
large_comparison = pd.DataFrame(large_metric_records)
costs = [item["api_usage"]["pilot_reported_cost_usd"] for item in large_candidate_summary.values()]
large_summary = {
    "design": "post-hoc, disjoint-row, same-sequence larger follow-up",
    "position_percentiles": [percent for percent in range(5, 100, 10)],
    "position_index_rule": "floor(p * (n - 1))",
    "pilot_rows": len(large), "test_sequences": len(metadata_sequences),
    "overlap_with_original_24_rows": overlap,
    "lstm_alarm_rows": LARGE_ALARMS,
    "max_new_api_calls": 2 * MAX_LARGE_CALLS_PER_PROMPT,
    "new_api_calls": sum(item["new_api_calls"] for item in large_candidate_summary.values()),
    "cache_hits": sum(item["cache_hits"] for item in large_candidate_summary.values()),
    "total_pilot_reported_cost_usd": sum(costs) if all(isinstance(v, (int, float)) for v in costs) else "unavailable",
    "candidates": large_candidate_summary,
    "metrics": large_metric_records,
}
large_row_columns = ["sequence_id", "raw_row_index", "sample_position", "lstm_pred_seconds",
                     "rf_probability", "recent_lstm_delta", "recent_lstm_std", "history_count",
                     "lstm_alarm", "pipeline_alarm", "actual_fail_5000"]
for candidate in LARGE_QUESTIONS:
    large_row_columns += [f"{candidate}_jev_score", f"{candidate}_jev_status",
                          f"{candidate}_jev_called", f"{candidate}_cache_hit",
                          f"{candidate}_jev_keep", f"{candidate}_hybrid_alarm",
                          f"{candidate}_jev_error"]
large[large_row_columns].to_csv(LARGE_OUT / "pilot_rows.csv", index=False)
large_comparison.to_csv(LARGE_OUT / "metric_comparison.csv", index=False)
(LARGE_OUT / "summary.json").write_text(
    json.dumps(large_summary, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
display(large_comparison)
print({"new_api_calls": large_summary["new_api_calls"], "cache_hits": large_summary["cache_hits"],
       "total_reported_cost_usd": large_summary["total_pilot_reported_cost_usd"]})
print({name: {key: value[key] for key in ("removed_false_positives", "removed_true_positives",
       "jev_success_rows", "jev_failed_rows")} for name, value in large_candidate_summary.items()})
print("Same eight sequences and post-hoc prompts: this is exploratory, not independent generalization evidence.")

### 관찰

새 80행에서 LSTM 경보는 23건(TP 15, FP 8)이었다. 두 보수적 prompt 모두 23건을 전부 유지하여 true alarm 추가 제거는 0건이었지만 false alarm 제거도 0건이었다. 따라서 이 샘플에서 JEV confirmation gate는 LSTM 경보를 실제로 걸러내지 못했다. 이 결과는 이전 24행과 겹치지 않는 row에 대한 사후 확인이지만 같은 8개 sequence를 사용하며 prompt도 이전 결과를 보고 작성했다.